# Deep dive 12/13: `complaints`

Dictionary declares 80,000 rows, Source: PQR, Partition: daily. 26 Bronze columns; **Silver has 25** -- `origin_interaction_id` is intentionally dropped, per `table_specs.py`'s spec comment: the Bronze profile found it 100% null / 0 distinct across all 67,095 rows checked back then, a real drop candidate rather than a "keep just in case" column. Section 6 re-confirms that against the current data before trusting the old comment.

Dictionary NOT NULL: `complaint_id`, `creation_date`, `process_date`, `customer_id`, `case_type`, `category`, `reception_channel`, `description`, `priority`, `status`, `sla_breached`, `is_repeat_complainer` -- **12 columns**. `contracts.py`'s current `required` tuple has 10 -- **`description` and `is_repeat_complainer` are both missing**, checked in section 3 as fix candidates (following the established caution: only recommend adding a missing column to `required` if it's actually clean in real data, per the `call_transcripts.duration_seconds` lesson).

`claimed_amount`/`compensation_granted` are typed DOUBLE but **NOT converted to USD** here, unlike `products`/`transactions` -- per the spec comment, `currency` is 67.54% null on this table and there's no natural single "as of" date the way `transactions` has one. Section 7 checks this directly.

This table also has a natural **workflow sequence** (`creation_date` -> `assignment_date` -> `first_response_date` -> `resolution_date` -> `closing_date`) and a `status` field that should logically correlate with which of those dates are populated -- sections 10-11 check both.

Run top to bottom. Checklist near the end; free cells after that.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver side by side (excluding long text columns)

In [ ]:
sample_ids = q("SELECT complaint_id FROM bronze.complaints ORDER BY complaint_id LIMIT 20")["complaint_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

cols = ("complaint_id, creation_date, process_date, customer_id, case_type, category, subcategory, "
        "reception_channel, affected_product_id, related_branch_id, claimed_amount, currency, "
        "priority, status, assigned_agent_id, sla_breached, resolution_days, "
        "compensation_granted, resolution_satisfaction, is_repeat_complainer")
print("Bronze:")
display(q(f"SELECT {cols} FROM bronze.complaints WHERE complaint_id IN ({ids_sql}) ORDER BY complaint_id"))
print("Silver:")
display(q(f"SELECT {cols} FROM silver.fact_complaints WHERE complaint_id IN ({ids_sql}) ORDER BY complaint_id"))

## 1. Column inventory -- names AND types, Bronze vs. Silver (26 Bronze, 25 Silver -- `origin_interaction_id` dropped)

In [ ]:
expected_silver_columns = [
    "complaint_id", "creation_date", "process_date", "customer_id", "case_type", "category",
    "subcategory", "reception_channel", "affected_product_id", "related_branch_id", "description",
    "claimed_amount", "currency", "priority", "status", "assigned_agent_id", "assignment_date",
    "first_response_date", "resolution_date", "closing_date", "sla_breached", "resolution_days",
    "resolution", "compensation_granted", "resolution_satisfaction", "is_repeat_complainer",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_complaints'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_complaints:", missing or "none")
print("Extra in silver.fact_complaints:", extra or "none")
print()
print(f"Total expected: {len(expected_silver_columns)}, total actual: {len(actual_silver_columns)}")
print("origin_interaction_id correctly absent from Silver:", "origin_interaction_id" not in actual_silver_columns)
print("Column order as built:", actual_silver_columns)

In [ ]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='complaints' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_complaints' ORDER BY ordinal_position
"""))

## 2. Row-count reconciliation + dedup

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.complaints")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT complaint_id) AS n FROM bronze.complaints")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_complaints")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT complaint_id) AS n FROM silver.fact_complaints")["n"][0]

print(f"bronze.complaints:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct complaint_id")
print(f"silver.fact_complaints: {silver_total:,} rows, {silver_distinct_pk:,} distinct complaint_id")
print()
print("Dictionary declares 80,000 rows -- matches" if bronze_total == 80_000 else f"Dictionary declares 80,000 rows -- MISMATCH, Bronze has {bronze_total:,}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 12 dictionary-declared columns

Including `description` and `is_repeat_complainer`, both missing from `contracts.py`'s `required` tuple -- checked here as fix candidates.

In [ ]:
declared_not_null = ["complaint_id", "creation_date", "process_date", "customer_id", "case_type",
                      "category", "reception_channel", "description", "priority", "status",
                      "sla_breached", "is_repeat_complainer"]
currently_required = {"complaint_id","creation_date","process_date","customer_id","case_type",
                       "category","reception_channel","priority","status","sla_breached"}

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.complaints
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_complaints WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "in_contracts_required": col in currently_required,
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

## 4. Nullable columns -- population rate, Bronze vs. Silver

In [ ]:
nullable_cols = ["subcategory", "affected_product_id", "related_branch_id", "claimed_amount",
                  "currency", "assigned_agent_id", "assignment_date", "first_response_date",
                  "resolution_date", "closing_date", "resolution_days", "resolution",
                  "compensation_granted", "resolution_satisfaction"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.complaints")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.complaints
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_complaints WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

## 5. Domain checks -- `case_type`, `reception_channel`, `priority`, `status`

In [ ]:
# case_type -- dictionary: Complaint, Claim, Request, Suggestion
print("case_type:")
display(q("SELECT case_type, count(*) AS n FROM bronze.complaints GROUP BY 1 ORDER BY n DESC"))
# reception_channel -- dictionary: Call Center, Email, Web, App, Branch, Regulator
print("reception_channel:")
display(q("SELECT reception_channel, count(*) AS n FROM bronze.complaints GROUP BY 1 ORDER BY n DESC"))
# priority -- dictionary: Low, Medium, High, Critical
print("priority:")
display(q("SELECT priority, count(*) AS n FROM bronze.complaints GROUP BY 1 ORDER BY n DESC"))
# status -- dictionary text was overlapping/truncated in the PDF: Open, In Process, Escalated, Resolved, Closed, Rejected(?)
print("status:")
display(q("SELECT status, count(*) AS n FROM bronze.complaints GROUP BY 1 ORDER BY n DESC"))

## 6. `origin_interaction_id` -- re-confirming the Silver drop decision, plus the other 4 FKs reproduced directly

`origin_interaction_id` doesn't exist in `silver.fact_complaints` (dropped by design), so its own check runs against Bronze only -- confirming it's still 100% null / 0 distinct before trusting the old spec comment on the current data.

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.complaints")["n"][0]
origin_populated = q("""
    SELECT count(*) AS n FROM bronze.complaints
    WHERE origin_interaction_id IS NOT NULL AND trim(origin_interaction_id) <> ''
""")["n"][0]
origin_distinct = q("SELECT count(DISTINCT origin_interaction_id) AS n FROM bronze.complaints")["n"][0]
print(f"origin_interaction_id: {origin_populated}/{bronze_total} populated, {origin_distinct} distinct non-null values")
print("Drop decision still holds -- fully null/uninformative" if origin_populated == 0 else "NOT fully null anymore -- re-investigate the drop")

In [ ]:
fk_checks = [
    ("customer_id", "silver.dim_customers", "customer_id"),
    ("affected_product_id", "silver.dim_products", "product_id"),
    ("related_branch_id", "silver.dim_branches", "branch_id"),
    ("assigned_agent_id", "silver.dim_service_agents", "agent_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_complaints WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_complaints t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

## 7. `claimed_amount`/`currency`/`compensation_granted` -- confirming NOT converted to USD, currency domain

Per the spec comment: no `amount_usd`-style column exists here, unlike `products`/`transactions`. Confirming that's really true (no hidden USD column), and checking `currency`'s real domain.

In [ ]:
print("Silver columns containing 'usd' (expect none):")
display(q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_complaints' AND column_name ILIKE '%usd%'
"""))
print()
print("currency domain (Bronze, nullable, expect ~67% null per prior spec comment):")
display(q("SELECT currency, count(*) AS n FROM bronze.complaints GROUP BY 1 ORDER BY n DESC"))
print()
print("claimed_amount / compensation_granted range sanity:")
display(q("""
    SELECT
        min(claimed_amount) AS min_claimed, max(claimed_amount) AS max_claimed,
        min(compensation_granted) AS min_comp, max(compensation_granted) AS max_comp,
        count(*) FILTER (WHERE claimed_amount < 0) AS negative_claimed,
        count(*) FILTER (WHERE compensation_granted < 0) AS negative_comp,
        count(*) FILTER (WHERE compensation_granted > claimed_amount) AS comp_exceeds_claimed
    FROM silver.fact_complaints
"""))

## 8. `resolution_satisfaction` -- range sanity (dictionary: 1-5)

In [ ]:
q("""
    SELECT min(resolution_satisfaction) AS min_v, max(resolution_satisfaction) AS max_v,
           count(DISTINCT resolution_satisfaction) AS distinct_values,
           count(*) FILTER (WHERE resolution_satisfaction NOT BETWEEN 1 AND 5) AS out_of_range
    FROM silver.fact_complaints
    WHERE resolution_satisfaction IS NOT NULL
""")

## 9. `sla_breached`/`is_repeat_complainer` -- boolean sanity, cross-tab against `priority`/`status`

In [ ]:
print("sla_breached vs. priority:")
display(q("""
    SELECT priority, sla_breached, count(*) AS n
    FROM silver.fact_complaints GROUP BY 1, 2 ORDER BY 1, 2
"""))
print()
print("is_repeat_complainer value counts:")
display(q("SELECT is_repeat_complainer, count(*) AS n FROM silver.fact_complaints GROUP BY 1"))

## 10. Workflow date sequencing -- `creation_date` -> `assignment_date` -> `first_response_date` -> `resolution_date` -> `closing_date`

Checking the dates arrive in a logical order, since this is a natural business-process sequence.

In [ ]:
q("""
    SELECT
        count(*) FILTER (WHERE assignment_date IS NOT NULL AND assignment_date < creation_date) AS assignment_before_creation,
        count(*) FILTER (WHERE first_response_date IS NOT NULL AND assignment_date IS NOT NULL AND first_response_date < assignment_date) AS response_before_assignment,
        count(*) FILTER (WHERE resolution_date IS NOT NULL AND first_response_date IS NOT NULL AND resolution_date < first_response_date) AS resolution_before_response,
        count(*) FILTER (WHERE closing_date IS NOT NULL AND resolution_date IS NOT NULL AND closing_date < resolution_date) AS closing_before_resolution
    FROM silver.fact_complaints
""")

In [ ]:
# resolution_days consistency: does it roughly match (resolution_date - creation_date) in days?
q("""
    SELECT
        count(*) AS total_with_both,
        avg(abs(resolution_days - date_diff('day', creation_date, resolution_date))) AS avg_abs_diff_days,
        max(abs(resolution_days - date_diff('day', creation_date, resolution_date))) AS max_abs_diff_days,
        count(*) FILTER (WHERE resolution_days <> date_diff('day', creation_date, resolution_date)) AS exact_mismatches
    FROM silver.fact_complaints
    WHERE resolution_days IS NOT NULL AND resolution_date IS NOT NULL
""")

## 11. `status` vs. date population -- does the workflow stage line up with which dates are filled in?

E.g. does `status = 'Resolved'`/`'Closed'` correlate with `resolution_date` being populated, and does `status = 'Open'` correlate with the later dates being empty?

In [ ]:
q("""
    SELECT status,
        count(*) AS n,
        round(100.0 * count(*) FILTER (WHERE assignment_date IS NOT NULL) / count(*), 1) AS pct_has_assignment,
        round(100.0 * count(*) FILTER (WHERE first_response_date IS NOT NULL) / count(*), 1) AS pct_has_first_response,
        round(100.0 * count(*) FILTER (WHERE resolution_date IS NOT NULL) / count(*), 1) AS pct_has_resolution,
        round(100.0 * count(*) FILTER (WHERE closing_date IS NOT NULL) / count(*), 1) AS pct_has_closing
    FROM silver.fact_complaints
    GROUP BY 1
    ORDER BY 1
""")

## 12. Cross-table consistency -- `affected_product_id`'s owner vs. `customer_id`

Reusing the pattern `checks.py`'s own `relationship_checks()` already runs (`product_owner_mismatch`) -- reproducing it directly here rather than trusting it blindly.

In [ ]:
q("""
    SELECT
        count(*) AS total_linked,
        count(*) FILTER (WHERE c.customer_id = p.customer_id) AS owner_matches,
        count(*) FILTER (WHERE c.customer_id <> p.customer_id) AS owner_mismatches
    FROM silver.fact_complaints c
    JOIN silver.dim_products p ON p.product_id = c.affected_product_id
    WHERE c.affected_product_id IS NOT NULL AND c.customer_id IS NOT NULL
""")

## 13. `creation_date`/`process_date` -- cast correctness, rollback check, window check

In [ ]:
q("""
    SELECT
        typeof(creation_date) AS creation_date_type, typeof(process_date) AS process_date_type,
        min(creation_date) AS earliest_creation, max(creation_date) AS latest_creation,
        min(process_date) AS earliest_process, max(process_date) AS latest_process,
        count(*) AS total,
        count(*) FILTER (WHERE process_date <> CAST(creation_date AS DATE)) AS process_date_differs
    FROM silver.fact_complaints
    GROUP BY ALL
""")

In [ ]:
q("""
    SELECT
        count(*) FILTER (WHERE creation_date < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE creation_date > TIMESTAMP '2026-06-17 23:59:59') AS after_window
    FROM silver.fact_complaints
""")

## Checklist

- [ ] Column inventory (1): 25 Silver columns present (26 Bronze minus `origin_interaction_id`), types correct
- [ ] Row reconciliation (2): 80,000 rows as the dictionary declares, dedup clean
- [ ] NOT NULL audit (3): all 12 declared columns populated -- note `description`/`is_repeat_complainer` fix candidates
- [ ] Nullable columns (4): population rates make sense
- [ ] Domain checks (5): `case_type`/`reception_channel`/`priority`/`status` real values
- [ ] `origin_interaction_id` drop (6): re-confirmed still 100% null/0 distinct; other 4 FKs reproduced directly
- [ ] USD conversion (7): confirmed absent by design, `currency` domain sane
- [ ] `resolution_satisfaction` (8): within 1-5
- [ ] Booleans (9): `sla_breached`/`is_repeat_complainer` sane, cross-tab against `priority`
- [ ] Date sequencing (10): workflow dates arrive in logical order, `resolution_days` consistent
- [ ] Status vs. dates (11): `status` correlates with which workflow dates are populated
- [ ] Cross-table consistency (12): `affected_product_id`'s owner matches `customer_id`
- [ ] Dates (13): rollback rate, window check

In [ ]:
con.close()
print("closed")